# pkl 层二次排查 —— 逐问题自查 notebook

配套文档：`v3.1-DIAG-pkl层二次排查问题清单.md`（同目录）

**这份 notebook 做什么**：把清单里 A / B / C / D 四类的**每一条**都配上
① 统计代码（跑全库，出数字）② 案例代码（把出问题那几根 bar 原样打出来）。
E 类是真实市场结构不是错误，不在这里。

**怎么跑**：从上往下依次执行。第 1 节把 6 GB 的 pkl 读进内存（约 8 秒 / 峰值约 8 GB），
之后所有单元都复用同一个 `df`，不再重复读盘。

**判据一览**

| 类 | 条 | 判据 |
|---|---|---|
| A | A-1 | 整段 session 成交量恒为 0，且同日历日同段全市场也为 0 |
| A | A-2 | 整段除**最后一根**外全零，最后一根有量 → 再按「末根量 / 该品种夜盘量中位数」分两型 |
| A | A-3 | 日盘段成交量恒为 0，且 `trading_date` ≠ 日历日 |
| A | A-4 | 整段成交量为 0，但同日历日同段**全市场有成交** → 真实无人交易，**勿删** |
| B | B-1 | `K` 为 NaN 或 ≤ 0 |
| B | B-2 | `contract` 在交易日内部发生变更 |
| B | B-3 | `(underlying_symbol, datetime)` 重复 |
| B | B-4 | 同一 `contract` 内 `K` 取值多于 1 个 |
| ~~B-5~~ → **E-11** | `contract` 的字母前缀 ≠ `underlying_symbol` —— **核实后不是错误**，见该节 |
| C | C-1 | 品种末个交易日远早于全库末日，但末日持仓仍在 |
| C | C-2 | 品种在自身 [首日, 末日] 区间内缺市场交易日 |
| D | D-1 | `total_turnover` / `volume` / `open_interest` 出现负值 |
| D | D-2 | `volume==0 & turnover>0` 或 `volume>0 & turnover==0` |
| D | D-3 | OHLC 任一 ≤ 0 或 NaN |
| F | F-1 | 2023-05-26 起, 有夜盘品种是否有 end 在 09:00 的日盘竞价 bar |
| F | F-2 | 交易日第一根 bar (竞价 bar) 是否 `open==high==low==close` |

## 0 · 环境与路径

In [1]:
import os, sys, time, json
import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

# notebook 在 data/01-pkl层/二次排查/ 下, pkl 在上一级
HERE = os.getcwd()
CAND = [
    os.path.join(HERE, "..", "all_symbol_min_full_main_close_k_1.pkl"),
    os.path.join(HERE, "all_symbol_min_full_main_close_k_1.pkl"),
    r"D:\2026_Summer\TradingApp\data\01-pkl层\all_symbol_min_full_main_close_k_1.pkl",
]
PKL = next((os.path.abspath(p) for p in CAND if os.path.exists(p)), None)
assert PKL, "找不到 pkl, 手动把 PKL 改成绝对路径"
print("pkl :", PKL)
print("size:", f"{os.path.getsize(PKL)/1e9:.2f} GB")

# 当前 59 品种池 (v3.0-LEDG Step 1-4 的结果), 用来区分「全库口径」和「池内口径」
POOL = {"A","AG","AL","AP","AU","B","BC","BU","C","CF","CJ","CS","CU","CY","EB","EG","FG","FU",
        "HC","I","IC","IF","IH","J","JD","JM","L","LH","LU","M","MA","NI","NR","OI","P","PB",
        "PF","PG","PK","PP","RB","RM","RU","SA","SC","SF","SM","SN","SP","SR","SS","T","TA",
        "TF","TS","UR","V","Y","ZN"}
print("pool:", len(POOL))

# 交易所归属 (79 个品种)。F 类两条检查按交易所分组, 必须有这张表
EXCH = {}
for _e, _ss in [
    ("SHFE",  "CU AL ZN PB NI SN AU AG RB WR HC SS BU RU SP FU AO BR"),
    ("INE",   "SC LU NR BC EC"),
    ("DCE",   "A B C M Y P L V PP J JM I FB BB JD EG RR EB PG LH CS"),
    ("ZCE",   "SR CF TA OI RM MA FG RS RI WH PM ZC JR LR SF SM CY AP CJ UR SA PF PK SH PX"),
    ("CFFEX", "IF IC IH IM T TF TS TL"),
    ("GFEX",  "SI LC"),
]:
    for _s in _ss.split():
        EXCH[_s] = _e
print("exchanges:", {k: sum(1 for v in EXCH.values() if v == k)
                     for k in ["SHFE","INE","DCE","ZCE","CFFEX","GFEX"]})

pkl : d:\2026_Summer\TradingApp\data\01-pkl层\all_symbol_min_full_main_close_k_1.pkl
size: 5.96 GB
pool: 59
exchanges: {'SHFE': 18, 'INE': 5, 'DCE': 21, 'ZCE': 25, 'CFFEX': 8, 'GFEX': 2}


## 1 · 载入 pkl，核对基础事实

先把「这份文件是什么」钉死。下面 5 个数字是后面所有结论的地基，
任何一个对不上，说明拿到的不是同一份交付文件，后面的数字都不用比了。

In [2]:
t0 = time.time()
df = pd.read_pickle(PKL)
print(f"load {time.time()-t0:.1f}s")

df.index.name = "datetime"
dt  = df.index
sym = df["underlying_symbol"].astype("category")

# 派生列: 分钟序号 / 日历日 / 段(0=日盘 1=夜盘)
mod = (dt.hour * 60 + dt.minute).to_numpy().astype(np.int32)      # 09:01 -> 541
cal = dt.normalize().to_numpy().astype("datetime64[D]")
td  = df["trading_date"].to_numpy().astype("datetime64[D]")
seg = ((mod >= 21*60) | (mod <= 179)).astype(np.int8)             # 夜盘 = 21:00-02:59

print("shape          :", df.shape)
print("columns        :", list(df.columns))
print("symbols        :", sym.cat.categories.size)
print("time span      :", dt.min(), "->", dt.max())
print("trading dates  :", pd.unique(td).size)
print("mem (GB)       :", round(df.memory_usage(deep=False).sum()/1e9, 2))

load 6.2s
shape          : (66284866, 11)
columns        : ['trading_date', 'close', 'open', 'high', 'low', 'volume', 'total_turnover', 'open_interest', 'contract', 'underlying_symbol', 'K']
symbols        : 79
time span      : 2010-01-04 09:00:00 -> 2026-07-29 15:15:00
trading dates  : 4017
mem (GB)       : 6.36


In [3]:
# 与二次排查文档对照 —— 全部应为 True
CHECK = {
    "行数 == 66,284,866"        : len(df) == 66_284_866,
    "品种 == 79"                : sym.cat.categories.size == 79,
    "交易日 == 4,017"           : pd.unique(td).size == 4_017,
    "起点 == 2010-01-04 09:00"  : str(dt.min()) == "2010-01-04 09:00:00",
    "终点 == 2026-07-29 15:15"  : str(dt.max()) == "2026-07-29 15:15:00",
}
for k, v in CHECK.items():
    print(("  OK  " if v else " FAIL ") + k)

  OK  行数 == 66,284,866
  OK  品种 == 79
  OK  交易日 == 4,017
  OK  起点 == 2010-01-04 09:00
  OK  终点 == 2026-07-29 15:15


## 2 · 会话级聚合表 `SESS`

A 类四条全部建立在这张表上，所以单独建一次、后面反复用。

**分段键 = (品种, trading_date, 日历日, 段)**，四个都不能少：

- 少了**日历日** → 元旦那种「伪造日盘 + 真实日盘同属一个 trading_date」会被真实成交掩盖（A-3 就查不出来）
- 少了**段** → 日盘的成交量会盖住同一 trading_date 里被伪造的夜盘

另外给每个夜盘 session 算一个 `anchor`（该夜盘所在的**晚间日历日**），
因为 21:00–02:30 的夜盘横跨两个日历日，要用 anchor 才能把它们并回一段。

In [4]:
t0 = time.time()
sc   = sym.cat.codes.to_numpy()
SYMS = np.asarray(sym.cat.categories)
SYMV = SYMS[sc]          # 逐行的品种名, 只物化这一次 (66M 行的 object 数组很贵, 别反复建)

vol = df["volume"].to_numpy()
tov = df["total_turnover"].to_numpy()
oi  = df["open_interest"].to_numpy()
op, hi, lo, cl = (df[c].to_numpy() for c in ("open", "high", "low", "close"))
K   = df["K"].to_numpy()
ctr = df["contract"].to_numpy()

pad_bar = (vol == 0) & (tov == 0) & (op == hi) & (hi == lo) & (lo == cl)

DAY0  = np.datetime64("2000-01-01")
td_i  = (td  - DAY0).astype(np.int64)
cal_i = (cal - DAY0).astype(np.int64)
# 整数拼键: 6,600 万行上做字符串拼接会慢到分钟级
raw_key       = ((sc.astype(np.int64) * 20000 + td_i) * 20000 + cal_i) * 2 + seg
code_, uniq_  = pd.factorize(raw_key)
n_seg         = uniq_.size

def agg_sum(x):   return np.bincount(code_, weights=x, minlength=n_seg)
def agg_first(x):
    out = np.empty(n_seg, dtype=x.dtype); out[code_[::-1]] = x[::-1]; return out
def agg_last(x):
    out = np.empty(n_seg, dtype=x.dtype); out[code_] = x; return out

SESS = pd.DataFrame({
    "sym"      : SYMS[(uniq_ // (2 * 20000 * 20000)).astype(int)],
    "td"       : (DAY0 + ((uniq_ // (2*20000)) % 20000).astype("timedelta64[D]")),
    "cal"      : (DAY0 + ((uniq_ // 2) % 20000).astype("timedelta64[D]")),
    "seg"      : (uniq_ % 2).astype(np.int8),
    "bars"     : np.bincount(code_, minlength=n_seg),
    "vol"      : agg_sum(vol),
    "pad"      : agg_sum(pad_bar.astype(float)).astype(np.int64),
    "vol_last" : agg_last(vol),
    "mod_first": agg_first(mod),
    "mod_last" : agg_last(mod),
    "oi_first" : agg_first(oi),
    "oi_last"  : agg_last(oi),
    "hi_last"  : agg_last(hi),
    "lo_last"  : agg_last(lo),
})
# 夜盘 anchor: 整段落在凌晨的, 归到前一个日历日
SESS["anchor"] = np.where((SESS.seg == 1) & (SESS.mod_last <= 179),
                          SESS.cal - pd.Timedelta("1D"), SESS.cal)
# 把横跨午夜的夜盘并成一段
NIGHT = (SESS[SESS.seg == 1]
         .sort_values(["sym", "td", "cal"])
         .groupby(["sym", "td", "anchor"], as_index=False)
         .agg(bars=("bars","sum"), vol=("vol","sum"), pad=("pad","sum"),
              vol_last=("vol_last","last"), mod_last=("mod_last","last"),
              oi_first=("oi_first","first"), oi_last=("oi_last","last"),
              hi_last=("hi_last","last"), lo_last=("lo_last","last")))
DAY = SESS[SESS.seg == 0].copy()

print(f"SESS {len(SESS):,} 段   NIGHT {len(NIGHT):,}   DAY {len(DAY):,}   {time.time()-t0:.1f}s")
SESS.head(3)

SESS 347,409 段   NIGHT 108,839   DAY 208,538   4.1s


,sym,td,cal,seg,bars,vol,pad,vol_last,mod_first,mod_last,oi_first,oi_last,hi_last,lo_last,anchor
0,A,2010-01-04,2010-01-04,0,226,160919.0,0,2872.0,540,900,138699.0,142148.0,4066.0,4056.0,2010-01-04
1,A,2010-01-05,2010-01-05,0,226,126820.0,0,1093.0,540,900,142234.0,141692.0,4067.0,4065.0,2010-01-05
2,A,2010-01-06,2010-01-06,0,226,430406.0,0,4426.0,540,900,141699.0,176415.0,4157.0,4153.0,2010-01-06


In [5]:
# 全市场活动量: 用来区分「整个市场没开盘(伪造)」和「这个品种自己没人交易(真实)」
MKT_NIGHT = NIGHT.groupby("anchor").vol.sum()
MKT_DAY   = DAY.groupby("cal").vol.sum()
NIGHT["mkt_vol"] = NIGHT.anchor.map(MKT_NIGHT)
DAY["mkt_vol"]   = DAY.cal.map(MKT_DAY)
print("夜盘 anchor 日历日:", len(MKT_NIGHT), " 其中全市场零成交:", int((MKT_NIGHT == 0).sum()))
print("日盘   日历日     :", len(MKT_DAY),   " 其中全市场零成交:", int((MKT_DAY   == 0).sum()))

夜盘 anchor 日历日: 3053  其中全市场零成交: 17
日盘   日历日     : 4020  其中全市场零成交: 3


---
# A 类 · 伪造填补

生成了市场上不存在的 K 线。**共同判据：整段 session 除最后一根外全部零成交。**
只用「整段成交量 == 0」会漏掉 A-2 的 6 万根。

## A-1 · 长假前被取消的夜盘补成零成交 bar

**判据**：夜盘 session 成交量恒为 0。再用「同一晚间日历日、全市场夜盘成交量是否也为 0」
把「交易所整晚没开」和「这个品种自己没人交易（A-4）」分开。

**机制（本轮向上游核实后确定）** —— 以国庆为例，节前最后交易日 09-30、节后首日 10-09：

| | 09-30 那一夜 | 10-09 之前那一夜 |
|---|---|---|
| 交易所实际 | 长假前夜盘取消，**本来就不该有** | 无 |
| txt（2022-09-22 换表头**之前**） | 15:00 收盘后就没数据 —— **正确** | 无 —— **正确** |
| txt（换表头**之后**） | 同样正确 | **凭空补出 10-08 21:00 – 10-09 02:30** |
| Level-1 | 21:00–00:00 **有数据**，但节前本来无夜盘 → 判为 **Level-1 侧的问题** | 无 |

根因是「上一交易日」从**查交易日历**退化成了**按日历日往前推**（−1 天、周一 −3 天）。
诉求只有一条：**停止凭空补段**。**不需要补夜盘** —— 长假前交易所本来就不开夜盘。

**关于 2,455 / 423,942 这两个数**：能不能复现出来，取决于夜盘 session 用什么键分组。

- 按 **(品种, trading_date)** 分组 → **2,455 段 / 423,942 根**，与 v3.0-SPEC 逐字相同
- 按 **(品种, trading_date, 晚间日历日 anchor)** 分组 → **2,599 段 / 431,764 根**

差的 144 段来自元旦：那几天**一个 trading_date 底下挂着两个不同的夜盘**
（伪造的那个 + 真实的那个），按 trading_date 分组会把它们合成一段，
真实那半边的成交量把伪造那半边盖住了，于是漏掉。**anchor 才是对的键。**

In [6]:
# 口径 1: 按 (品种, trading_date) —— v3.0-SPEC 的口径
by_td = NIGHT.groupby(["sym","td"], as_index=False).agg(bars=("bars","sum"), vol=("vol","sum"))
z_td  = by_td[by_td.vol == 0]
# 口径 2: 按 (品种, trading_date, anchor) —— 把同一 trading_date 下的两个夜盘分开
a1    = NIGHT[NIGHT.vol == 0]

print(f"[A-1] 按 (品种,trading_date)        : {len(z_td):,} 段 / {int(z_td.bars.sum()):,} 根"
      f"   <- v3.0-SPEC 记的是 2,455 / 423,942 -> "
      f"{'一致' if (len(z_td), int(z_td.bars.sum())) == (2455, 423942) else '不一致'}")
print(f"      按 (品种,trading_date,anchor) : {len(a1):,} 段 / {int(a1.bars.sum()):,} 根")
print(f"      按 trading_date 分组漏掉的     : {len(a1)-len(z_td):,} 段 / "
      f"{int(a1.bars.sum()-z_td.bars.sum()):,} 根")
miss = a1.merge(z_td[["sym","td"]].assign(_hit=1), on=["sym","td"], how="left")
miss = miss[miss._hit.isna()]
print(f"      漏掉的那批落在哪几个交易日     : {sorted(set(miss.td.dt.strftime('%Y-%m-%d')))}")

fab  = a1[a1.mkt_vol == 0]      # 全市场当晚也零成交 -> 交易所整晚没开
real = a1[a1.mkt_vol >  0]      # 别的品种在交易 -> 这个品种自己没人交易, 归 A-4
print(f"      其中 全市场同晚零成交(伪造): {len(fab):,} 段 / {int(fab.bars.sum()):,} 根")
print(f"           全市场同晚有成交(A-4) : {len(real):,} 段 / {int(real.bars.sum()):,} 根")
print(f"      池内(59)口径             : {len(a1[a1.sym.isin(POOL)]):,} 段 / "
      f"{int(a1[a1.sym.isin(POOL)].bars.sum()):,} 根")

print("\n伪造夜盘的日历日清单 (全部):")
print(fab.groupby(fab.anchor.dt.strftime("%Y-%m-%d"))
        .agg(品种数=("sym","nunique"), bar数=("bars","sum")).to_string())

[A-1] 按 (品种,trading_date)        : 2,455 段 / 423,942 根   <- v3.0-SPEC 记的是 2,455 / 423,942 -> 一致
      按 (品种,trading_date,anchor) : 2,599 段 / 431,764 根
      按 trading_date 分组漏掉的     : 144 段 / 7,822 根
      漏掉的那批落在哪几个交易日     : ['2023-01-03', '2024-01-02', '2025-01-02']
      其中 全市场同晚零成交(伪造): 700 段 / 109,653 根
           全市场同晚有成交(A-4) : 1,899 段 / 322,111 根
      池内(59)口径             : 1,704 段 / 251,323 根

伪造夜盘的日历日清单 (全部):
            品种数  bar数
anchor               
2013-07-04    2   662
2013-12-31    6  1626
2015-09-25   27  5217
2019-12-31   43  6673
2022-10-07   47  7277
2022-12-30   47  7277
2023-01-27   47  7277
2023-06-23   47  7277
2023-10-06   50  7640
2023-12-29   50  7640
2024-02-16   50  7640
2024-04-05   50  7640
2024-05-03   47  7277
2024-12-31   47  6820
2025-04-04   47  7277
2026-01-02   47  7277
2026-06-19   46  7156


In [7]:
# 案例: RB 交易日 2024-10-08 (国庆后首个交易日), 10-07 晚间的夜盘整段零成交
ex = df[(sym == "RB") & (df.trading_date == "2024-10-08")]
ex_night = ex[(ex.index.hour >= 21) | (ex.index.hour <= 2)]
print("RB td=2024-10-08 夜盘:", len(ex_night), "根, 成交量合计", ex_night.volume.sum())
print(ex_night[["open","high","low","close","volume","total_turnover","open_interest"]]
      .iloc[[0,1,-2,-1]].to_string())
ex_day = ex[(ex.index.hour >= 9) & (ex.index.hour <= 15)]
print(f"\n同日日盘(真实): {len(ex_day)} 根, 成交量合计 {ex_day.volume.sum():,.0f}, "
      f"首根 {ex_day.index[0]:%H:%M} open={ex_day.open.iloc[0]}")

RB td=2024-10-08 夜盘: 121 根, 成交量合计 0.0
                       open    high     low   close  volume  total_turnover  open_interest
datetime                                                                                  
2024-10-07 21:00:00  3551.0  3551.0  3551.0  3551.0     0.0             0.0      1622556.0
2024-10-07 21:01:00  3551.0  3551.0  3551.0  3551.0     0.0             0.0      1622556.0
2024-10-07 22:59:00  3551.0  3551.0  3551.0  3551.0     0.0             0.0      1622556.0
2024-10-07 23:00:00  3551.0  3551.0  3551.0  3551.0     0.0             0.0      1622556.0

同日日盘(真实): 225 根, 成交量合计 4,220,545, 首根 09:01 open=3690.0


## A-2 · 整段除末根外全零，末根带量

**这是 A-1 判据的盲区**：因为末根有量，「整段成交量 == 0」判不出来。

跑完会发现这些段其实是**两种性质完全不同的东西**：

| 型 | 是什么 | 上游核实结果 | 怎么办 |
|---|---|---|---|
| **A 型** | 夜盘确实被取消了，末根装的是**次日早盘集合竞价**（量很小） | txt 与 pkl 逐字一致；**Level-1 是逐笔推流，竞价在那边是独立的一笔** → 错标发生在 txt 合成环节 | 提需求把竞价归回 09:00；在此之前整段删 |
| **B 型** | **夜盘真实发生过**，但整段成交被压扁进末根、其余填平 K 线、OI 冻结 | txt 侧同形（也是只有末根有量、量极大）→ 不是 pkl 造的；**Level-1 不覆盖 2014-12，源头无从考证** | **不修** —— 分钟结构永久丢失，只剔这些夜盘段 |

**分型要两个特征，一个不够**：
- `ratio` = 末根成交量 / 该品种夜盘成交量中位数 —— A 型 ~0.005，B 型 ~1.0
- `OI冻结` = 整夜持仓量一动没动 —— B 型的夜盘是假的分钟结构，OI 停在前收

单看 `ratio` 有 11 段落在 0.02–0.20 的灰带里判不了；**加上 `OI冻结` 就干净了**。
下面会把灰带 11 段逐条打出来，不藏。

In [8]:
# 整段除末根外全零 + 末根有量
cand = NIGHT[(NIGHT.bars >= 60) & ((NIGHT.vol - NIGHT.vol_last) == 0) & (NIGHT.vol_last > 0)].copy()
med  = NIGHT[NIGHT.vol > 0].groupby("sym").vol.median()
cand["ratio"]    = cand.vol_last / cand.sym.map(med)
cand["末根平K"]  = cand.hi_last == cand.lo_last
cand["OI冻结"]   = cand.oi_first == cand.oi_last
# 分型: 量级够大, 或者「OI 冻结 + 量级不算小」
is_B = (cand.ratio > 0.20) | (cand.OI冻结 & (cand.ratio > 0.05))
cand["型"]       = np.where(is_B, "B_夜盘被压扁成一根", "A_取消夜盘+末根装竞价")
cand["夜盘收盘"] = cand.mod_last.map(lambda m: f"{m//60:02d}:{m%60:02d}")

print(f"[A-2] 合计 {len(cand)} 段 / {int(cand.bars.sum()):,} 根 / {cand.sym.nunique()} 个品种"
      f"  (池内 {int(cand.sym.isin(POOL).sum())} 段)")

print("\nratio 分档 x OI冻结 —— 灰带靠 OI冻结 拆开:")
cand["档"] = pd.cut(cand.ratio, [0, 0.02, 0.20, 1e9], labels=["<0.02", "0.02-0.20 灰带", ">0.20"])
print(pd.crosstab(cand["档"], cand["OI冻结"]).to_string())

print("\n灰带 11 段逐条 (OI冻结=True 的判成 B 型):")
print(cand[(cand.ratio >= 0.02) & (cand.ratio <= 0.20)]
        .sort_values("ratio")[["sym","td","bars","vol_last","ratio","夜盘收盘","OI冻结","型"]]
        .to_string(index=False))

print("\n按型汇总:")
print(cand.groupby("型").agg(段数=("sym","size"), bar数=("bars","sum"),
        品种=("sym", lambda x: " ".join(sorted(set(x)))),
        比值中位=("ratio", lambda x: round(x.median(), 4)),
        末根平K=("末根平K","sum"), OI冻结=("OI冻结","sum"),
        起=("td", lambda x: f"{x.min():%Y-%m-%d}"),
        止=("td", lambda x: f"{x.max():%Y-%m-%d}")).T.to_string())

[A-2] 合计 232 段 / 54,963 根 / 17 个品种  (池内 232 段)

ratio 分档 x OI冻结 —— 灰带靠 OI冻结 拆开:
OI冻结          False  True 
档                         
<0.02           150      6
0.02-0.20 灰带      9      2
>0.20             1     64

灰带 11 段逐条 (OI冻结=True 的判成 B 型):
sym         td  bars  vol_last    ratio  夜盘收盘  OI冻结            型
 AL 2023-04-06   241    1082.0 0.022893 01:00 False A_取消夜盘+末根装竞价
 PB 2024-10-08   241     351.0 0.023702 01:00 False A_取消夜盘+末根装竞价
 AL 2025-01-02   302    1203.0 0.025453 01:00 False A_取消夜盘+末根装竞价
 AL 2024-10-08   241    1280.0 0.027082 01:00 False A_取消夜盘+末根装竞价
 PB 2025-01-02   302     427.0 0.028834 01:00 False A_取消夜盘+末根装竞价
 PB 2023-04-06   241     468.0 0.031602 01:00 False A_取消夜盘+末根装竞价
 PB 2024-06-11   241     508.0 0.034303 01:00 False A_取消夜盘+末根装竞价
 AG 2025-05-06   331   12226.0 0.039090 02:30 False A_取消夜盘+末根装竞价
 SN 2023-05-04   241     775.0 0.041099 01:00 False A_取消夜盘+末根装竞价
 CF 2014-12-22   151   14620.0 0.175409 23:30  True   B_夜盘被压扁成一根
 MA 2014-12-15   151   63788.0 0.19705

In [9]:
print("=== A 型 (取消夜盘 + 末根装次日竞价) 逐品种 ===")
A2a = cand[cand["型"].str.startswith("A")]
print(A2a.groupby("sym").agg(段数=("td","size"), bar数=("bars","sum"),
        夜盘收盘=("夜盘收盘", lambda x: sorted(set(x))),
        比值中位=("ratio", lambda x: round(x.median(), 4))).to_string())
print("\n涉及的交易日:", sorted(set(A2a.td.dt.strftime("%Y-%m-%d"))))

=== A 型 (取消夜盘 + 末根装次日竞价) 逐品种 ===
     段数  bar数     夜盘收盘    比值中位
sym                           
AG   15  5116  [02:30]  0.0057
AL   15  3676  [01:00]  0.0100
AU   15  5116  [02:30]  0.0024
B     1   151  [23:30]  0.0000
BC   14  3435  [01:00]  0.0012
CU   15  3676  [01:00]  0.0062
NI   15  3676  [01:00]  0.0016
PB   15  3676  [01:00]  0.0154
SC   15  5116  [02:30]  0.0017
SN   15  3676  [01:00]  0.0065
SS   15  3676  [01:00]  0.0034
ZN   15  3676  [01:00]  0.0041

涉及的交易日: ['2015-06-25', '2023-01-03', '2023-04-06', '2023-05-04', '2024-01-02', '2024-06-11', '2024-09-18', '2024-10-08', '2025-01-02', '2025-02-05', '2025-05-06', '2025-06-03', '2025-10-09', '2026-02-24', '2026-04-07', '2026-05-06']


In [10]:
# A 型案例: AG 交易日 2024-10-08
# 00:00-02:29 共 150 根零成交, 02:30 那根 close 恰等于当日 09:01 的 open
ex = df[(sym == "AG") & (df.trading_date == "2024-10-08")]
print("=== AG td=2024-10-08 ===")
print("00:00-02:30 段:")
print(ex.between_time("00:00", "02:30")[["open","high","low","close","volume","open_interest"]]
        .iloc[[0, 1, -3, -2, -1]].to_string())
d0 = ex.between_time("09:00", "15:00")
v230 = ex.between_time("02:30", "02:30")
print(f"\n日盘首根 {d0.index[0]:%H:%M}  open={d0.open.iloc[0]}  volume={d0.volume.iloc[0]:,.0f}")
print(f"02:30 那根 close  = {v230.close.iloc[0]}   与日盘 open 相等? "
      f"{v230.close.iloc[0] == d0.open.iloc[0]}")
print(f"02:30 那根 volume = {v230.volume.iloc[0]:,.0f}  vs 当日日盘合计 {d0.volume.sum():,.0f}"
      f"  = {v230.volume.iloc[0]/d0.volume.sum():.4%}   <- 竞价量级")

=== AG td=2024-10-08 ===
00:00-02:30 段:
                       open    high     low   close  volume  open_interest
datetime                                                                  
2024-10-08 00:00:00  7793.0  7793.0  7793.0  7793.0     0.0       430967.0
2024-10-08 00:01:00  7793.0  7793.0  7793.0  7793.0     0.0       430967.0
2024-10-08 02:28:00  7793.0  7793.0  7793.0  7793.0     0.0       430967.0
2024-10-08 02:29:00  7793.0  7793.0  7793.0  7793.0     0.0       430967.0
2024-10-08 02:30:00  7826.0  7826.0  7826.0  7826.0  1417.0       430373.0

日盘首根 09:01  open=7826.0  volume=13,707
02:30 那根 close  = 7826.0   与日盘 open 相等? True
02:30 那根 volume = 1,417  vs 当日日盘合计 739,418  = 0.1916%   <- 竞价量级


In [11]:
print("=== B 型 (真实夜盘被压扁成一根) 逐品种 ===")
A2b = cand[cand["型"].str.startswith("B")]
print(A2b.groupby("sym").agg(段数=("td","size"), bar数=("bars","sum"),
        夜盘收盘=("夜盘收盘", lambda x: sorted(set(x))),
        比值中位=("ratio", lambda x: round(x.median(), 2)),
        起止=("td", lambda x: f"{x.min():%Y-%m-%d} .. {x.max():%Y-%m-%d}")).to_string())
print(f"\n逐段明细 (全部 {len(A2b)} 段):")
print(A2b.sort_values(["sym","td"])[
        ["sym","td","bars","vol_last","ratio","夜盘收盘","末根平K","OI冻结"]].to_string(index=False))

=== B 型 (真实夜盘被压扁成一根) 逐品种 ===
     段数  bar数     夜盘收盘  比值中位                        起止
sym                                                   
BC    1   241  [01:00]  1.07  2024-01-02 .. 2024-01-02
CF   13  1963  [23:30]  0.38  2014-12-15 .. 2014-12-31
CU    1   241  [01:00]  2.31  2014-05-22 .. 2014-05-22
MA   13  1963  [23:30]  1.22  2014-12-15 .. 2014-12-31
RM   13  1963  [23:30]  1.37  2014-12-15 .. 2014-12-31
SR   13  1963  [23:30]  0.94  2014-12-15 .. 2014-12-31
TA   13  1963  [23:30]  0.84  2014-12-15 .. 2014-12-31

逐段明细 (全部 67 段):
sym         td  bars  vol_last    ratio  夜盘收盘  末根平K  OI冻结
 BC 2024-01-02   241    4868.0 1.074851 01:00  True False
 CF 2014-12-15   151   29801.0 0.357549 23:30  True  True
 CF 2014-12-16   151   16999.0 0.203952 23:30  True  True
 CF 2014-12-17   151  103679.0 1.243929 23:30  True  True
 CF 2014-12-18   151   52419.0 0.628917 23:30  True  True
 CF 2014-12-19   151   58478.0 0.701613 23:30  True  True
 CF 2014-12-22   151   14620.0 0.175409 23:30  True  

In [12]:
# B 型案例: CF 交易日 2014-12-17 —— 郑商所夜盘上线第 3 天
# 整夜 151 根里只有 23:30 那一根有量, 且它一个人扛了 103,679 手 (当日日盘才 87,809 手)
ex   = df[(sym == "CF") & (df.trading_date == "2014-12-17")]
nite = ex[(ex.index.hour >= 21) | (ex.index.hour <= 2)]
day  = ex.between_time("09:00", "15:00")
print("=== CF td=2014-12-17 (夜盘被压扁) ===")
print(f"夜盘 {len(nite)} 根, 有成交的 {int((nite.volume>0).sum())} 根, 夜盘成交量合计 {nite.volume.sum():,.0f}")
print(f"日盘 {len(day)} 根, 日盘成交量合计 {day.volume.sum():,.0f}")
print("\n夜盘首尾:")
print(nite[["open","high","low","close","volume","open_interest"]].iloc[[0,1,-2,-1]].to_string())
print(f"\n末根 high==low ? {nite.high.iloc[-1] == nite.low.iloc[-1]}"
      f"   整夜 OI 未动 ? {nite.open_interest.iloc[0] == nite.open_interest.iloc[-1]}"
      f"  ({nite.open_interest.iloc[0]:,.0f} -> {nite.open_interest.iloc[-1]:,.0f})")

ok  = df[(sym == "CF") & (df.trading_date == "2015-01-06")]
okn = ok[(ok.index.hour >= 21) | (ok.index.hour <= 2)]
print(f"\n对照 CF td=2015-01-06 (正常夜盘): {len(okn)} 根, 有成交 {int((okn.volume>0).sum())} 根, "
      f"合计 {okn.volume.sum():,.0f}, 末根仅 {okn.volume.iloc[-1]:,.0f}, "
      f"OI {okn.open_interest.iloc[0]:,.0f} -> {okn.open_interest.iloc[-1]:,.0f}")

=== CF td=2014-12-17 (夜盘被压扁) ===
夜盘 151 根, 有成交的 1 根, 夜盘成交量合计 103,679
日盘 225 根, 日盘成交量合计 87,809

夜盘首尾:
                        open     high      low    close    volume  open_interest
datetime                                                                        
2014-12-16 21:00:00  13020.0  13020.0  13020.0  13020.0       0.0       222143.0
2014-12-16 21:01:00  13020.0  13020.0  13020.0  13020.0       0.0       222143.0
2014-12-16 23:29:00  13020.0  13020.0  13020.0  13020.0       0.0       222143.0
2014-12-16 23:30:00  12840.0  12840.0  12840.0  12840.0  103679.0       222143.0

末根 high==low ? True   整夜 OI 未动 ? True  (222,143 -> 222,143)

对照 CF td=2015-01-06 (正常夜盘): 151 根, 有成交 151 根, 合计 26,433, 末根仅 298, OI 236,798 -> 237,741


In [13]:
# 「是不是只有头两周」—— 把 5 个郑商所品种在 23:30 收盘时代的每一个夜盘都过一遍
zce = ["CF","MA","RM","SR","TA"]
n23 = NIGHT[(NIGHT.sym.isin(zce)) & (NIGHT.mod_last == 23*60+30)].copy()
n23["压扁"] = (n23.bars >= 60) & ((n23.vol - n23.vol_last) == 0) & (n23.vol_last > 0)
n23["年"]   = n23.td.dt.year
print("=== CF/MA/RM/SR/TA 在 23:30 收盘时代的全部夜盘 ===")
print(n23.groupby("年").agg(夜盘段数=("sym","size"), 压扁段数=("压扁","sum")).to_string())
print(f"\n23:30 时代合计 {len(n23):,} 个夜盘, 被压扁的 {int(n23['压扁'].sum())} 个")
days = sorted(set(n23[n23['压扁']].td.dt.strftime('%Y-%m-%d')))
print(f"被压扁的交易日 ({len(days)} 个):", days)
print("\n一致性检查 —— 5 个品种是不是各自恰好中了同样多段:")
print(A2b[A2b.sym.isin(zce)].groupby("sym").td.nunique().to_string())
print("\n=> 只集中在 2014-12-15 ~ 2014-12-31 这 13 个交易日 (郑商所夜盘上线头两周),")
print("   5 个品种各 13 段, 一段不多一段不少;")
print("   2015 年起同一批品种的夜盘全部正常分布, 没有再出现。")
print("\n但同型缺陷在这一簇之外还有 2 例:")
print(A2b[~A2b.sym.isin(zce)][
        ["sym","td","bars","vol_last","ratio","夜盘收盘","OI冻结"]].to_string(index=False))

=== CF/MA/RM/SR/TA 在 23:30 收盘时代的全部夜盘 ===
      夜盘段数  压扁段数
年               
2014    65    65
2015  1185     0
2016  1185     0
2017  1190     0
2018  1180     0
2019  1115     0

23:30 时代合计 5,920 个夜盘, 被压扁的 65 个
被压扁的交易日 (13 个): ['2014-12-15', '2014-12-16', '2014-12-17', '2014-12-18', '2014-12-19', '2014-12-22', '2014-12-23', '2014-12-24', '2014-12-25', '2014-12-26', '2014-12-29', '2014-12-30', '2014-12-31']

一致性检查 —— 5 个品种是不是各自恰好中了同样多段:
sym
CF    13
MA    13
RM    13
SR    13
TA    13

=> 只集中在 2014-12-15 ~ 2014-12-31 这 13 个交易日 (郑商所夜盘上线头两周),
   5 个品种各 13 段, 一段不多一段不少;
   2015 年起同一批品种的夜盘全部正常分布, 没有再出现。

但同型缺陷在这一簇之外还有 2 例:
sym         td  bars  vol_last    ratio  夜盘收盘  OI冻结
 BC 2024-01-02   241    4868.0 1.074851 01:00 False
 CU 2014-05-22   241  109097.0 2.312259 01:00  True


## A-3 · 元旦被伪造成完整交易日

**判据**：日盘段成交量恒为 0，且该日历日**全市场**日盘成交量也为 0
（= 那天整个市场根本没开盘）。等价的独立判据：日盘 bar 的 `trading_date` ≠ 日历日。

**与 A-1 是同一个根因**（按日历日往前推、不查交易日历）：2023-01-02 那次就是 A-1 的形态，
2024-01-01 / 2025-01-01 那两次连**日盘**也一起补了。三例都是 txt 加工引入，不是 pkl 造的。

**预期**：144 段 / 33,300 根，只有 2023-01-02 / 2024-01-01 / 2025-01-01 三天。

In [14]:
a3 = DAY[(DAY.vol == 0) & (DAY.mkt_vol == 0)]
print(f"[A-3] 日盘整段零成交 且 全市场当日也零成交: {len(a3)} 段 / {int(a3.bars.sum()):,} 根")
print(a3.groupby(a3.cal.dt.strftime("%Y-%m-%d"))
        .agg(品种数=("sym","nunique"), bar数=("bars","sum"),
             对应交易日=("td", lambda x: x.iloc[0].strftime("%Y-%m-%d"))).to_string())

cross = DAY[DAY.td != DAY.cal]
print(f"\n交叉验证 —— 日盘段中 trading_date != 日历日: {len(cross)} 段 / {int(cross.bars.sum()):,} 根")
same = (set(map(tuple, a3[["sym","td","cal"]].astype(str).values.tolist()))
        == set(map(tuple, cross[["sym","td","cal"]].astype(str).values.tolist())))
print(f"两条独立判据是否给出同一集合: {same}")

[A-3] 日盘整段零成交 且 全市场当日也零成交: 144 段 / 33,300 根
            品种数   bar数       对应交易日
cal                               
2023-01-02   47  10575  2023-01-03
2024-01-01   50  11250  2024-01-02
2025-01-01   47  11475  2025-01-02

交叉验证 —— 日盘段中 trading_date != 日历日: 144 段 / 33,300 根
两条独立判据是否给出同一集合: True


In [15]:
# 案例: A 豆一 交易日 2023-01-03 的 bar 数是正常值的两倍
ex = df[(sym == "A") & (df.trading_date == "2023-01-03")]
print(f"A td=2023-01-03 共 {len(ex)} 根 (该品种正常日 346 根)")
g = ex.groupby(ex.index.normalize()).agg(bar数=("volume","size"), 成交量=("volume","sum"))
g["起"] = ex.groupby(ex.index.normalize()).apply(lambda x: x.index.min().strftime("%H:%M"))
g["止"] = ex.groupby(ex.index.normalize()).apply(lambda x: x.index.max().strftime("%H:%M"))
print(g.to_string())
print("\n伪造的 2023-01-02 日盘, 前 3 根:")
print(ex.loc["2023-01-02"].between_time("09:00","15:00")[
        ["open","high","low","close","volume","open_interest"]].head(3).to_string())

A td=2023-01-03 共 692 根 (该品种正常日 346 根)
            bar数       成交量      起      止
datetime                                
2022-12-30   121       0.0  21:00  23:00
2023-01-02   346       0.0  09:01  23:00
2023-01-03   225  104989.0  09:01  15:00

伪造的 2023-01-02 日盘, 前 3 根:
                       open    high     low   close  volume  open_interest
datetime                                                                  
2023-01-02 09:01:00  5179.0  5179.0  5179.0  5179.0     0.0       163939.0
2023-01-02 09:02:00  5179.0  5179.0  5179.0  5179.0     0.0       163939.0
2023-01-02 09:03:00  5179.0  5179.0  5179.0  5179.0     0.0       163939.0


## A-4 · 反例：僵尸品种真实无成交 —— **勿删**

跟 A-1 长得一模一样（整段零成交 + 平 K 线），但**同一日历日别的品种在正常交易**，
说明市场开着，只是这个品种没人交易。这类**不是伪造，删掉就是损坏数据**。

同一条判据还兜住**涨跌停封死**的整日零成交，例如 NI 2022-03-10 伦镍逼空。
只用「整段成交量 == 0」做清洗会把这一天整天删掉。

In [16]:
a4n = NIGHT[(NIGHT.vol == 0) & (NIGHT.mkt_vol > 0)]
a4d = DAY[(DAY.vol == 0) & (DAY.mkt_vol > 0)]
print(f"[A-4] 夜盘: {len(a4n):,} 段 / {int(a4n.bars.sum()):,} 根")
print(f"      日盘: {len(a4d):,} 段 / {int(a4d.bars.sum()):,} 根")
print("\n日盘零成交天数最多的品种 (全部是已剔除的僵尸品种):")
print(a4d.groupby("sym").td.nunique().sort_values(ascending=False).head(12).to_string())
print("\n池内(59) 仍会命中的品种:")
print(a4d[a4d.sym.isin(POOL)].groupby("sym").agg(
        天数=("td","nunique"), 起=("td", lambda x: f"{x.min():%Y-%m-%d}"),
        止=("td", lambda x: f"{x.max():%Y-%m-%d}")).to_string())

[A-4] 夜盘: 1,899 段 / 322,111 根
      日盘: 15,629 段 / 4,144,199 根

日盘零成交天数最多的品种 (全部是已剔除的僵尸品种):
sym
PM    2746
JR    2174
LR    2148
BB    1864
RI    1851
RS    1102
WH     891
ZC     813
FB     698
B      514
WR     369
SF     213

池内(59) 仍会命中的品种:
      天数           起           止
sym                             
B    514  2010-02-08  2017-10-12
BU     4  2014-12-15  2014-12-23
CY    46  2017-12-29  2018-05-10
NI     1  2022-03-10  2022-03-10
SC     1  2018-04-23  2018-04-23
SF   213  2014-12-25  2017-09-19
SM   194  2015-01-05  2016-06-01


In [17]:
# 案例: NI 2022-03-10 —— 真实封板, 整日零成交, 必须保留
ex = df[(sym == "NI") & (df.trading_date >= "2022-03-08") & (df.trading_date <= "2022-03-11")]
lbl = np.where((ex.index.hour >= 21) | (ex.index.hour <= 2), "夜", "日")
print(ex.groupby([ex.trading_date.dt.strftime("%Y-%m-%d"), lbl]).agg(
        bar数=("volume","size"), 成交量=("volume","sum"),
        最高=("high","max"), 最低=("low","min"), OI末=("open_interest","last")).to_string())
print(f"\n2022-03-10 当天全市场日盘成交量: {MKT_DAY.get(pd.Timestamp('2022-03-10')):,.0f}"
      "  -> 市场开着, 是 NI 自己封板")

                bar数      成交量        最高        最低       OI末
trading_date                                               
2022-03-08   夜   241  13577.0  228810.0  228810.0  147747.0
             日   225   2304.0  228810.0  228810.0  145656.0
2022-03-09   夜   241  25906.0  267700.0  267700.0  124412.0
             日   225  17812.0  267700.0  267700.0  114596.0
2022-03-10   夜   241      0.0  267700.0  267700.0  114596.0
             日   225      0.0  267700.0  267700.0  114596.0
2022-03-11   夜   241   4408.0  222190.0  222190.0  110756.0
             日   225    779.0  222190.0  222190.0  110521.0

2022-03-10 当天全市场日盘成交量: 12,047,090  -> 市场开着, 是 NI 自己封板


---
# B 类 · pkl 加工引入

`K` / `contract` / `underlying_symbol` 这三个字段 **txt 层根本没有**，
所以这几条缺陷只可能在 pkl 的加工环节产生，**上游无处可查，只能在 pkl 内自证**。

## B-1 · 复权因子 K 为 NaN 或 ≤ 0

`close/K` 是本数据集唯一正确的复权方式。K 缺失则无法复权；
**K 恰好等于 0 时 `close/K` 会静默产生 `inf`**，比 NaN 更危险。

In [18]:
b1 = pd.DataFrame({"sym": SYMV, "K": K}).groupby("sym").agg(
        总bar=("K","size"), K_NaN=("K", lambda x: int(x.isna().sum())),
        K_非正=("K", lambda x: int((x <= 0).sum())))
b1["不可用比例%"] = ((b1.K_NaN + b1.K_非正) / b1.总bar * 100).round(2)
bad = b1[(b1.K_NaN + b1.K_非正) > 0].sort_values("不可用比例%", ascending=False)
print("[B-1] K 有问题的品种:")
print(bad.to_string())
print(f"\nK 完全干净的品种: {len(b1) - len(bad)} / {len(b1)}   <- 没有中间地带")
print("池内(59)是否有 K 损坏:", sorted(set(bad.index) & POOL) or "无")
print("K 恰好等于 0 (会产生 inf) 的品种:", sorted(b1[b1.K_非正 > 0].index.tolist()))

[B-1] K 有问题的品种:
        总bar   K_NaN    K_非正  不可用比例%
sym                                 
ZC   1195605  487965    1384   40.93
JR    815634  313914       0   38.49
WH   1021294  343972     678   33.75
RS    714838  169274   15142   25.80
PM    909198       0  223514   24.58
RI    908972       0  223288   24.56

K 完全干净的品种: 73 / 79   <- 没有中间地带
池内(59)是否有 K 损坏: 无
K 恰好等于 0 (会产生 inf) 的品种: ['PM', 'RI', 'RS', 'WH', 'ZC']


In [19]:
# 案例: PM 的 K == 0, close/K 直接爆 inf
ex = df[(sym == "PM") & (df["K"] == 0)]
print(f"PM K==0 的 bar 数: {len(ex):,}")
print(ex[["close","K","contract","volume"]].head(3).to_string())
print("\n复权价 close/K =", (ex.close.iloc[:3] / ex.K.iloc[:3]).tolist())

PM K==0 的 bar 数: 223,514
                      close    K contract  volume
datetime                                         
2022-07-01 09:00:00  2911.0  0.0   PM2209     0.0
2022-07-01 09:01:00  2911.0  0.0   PM2209     0.0
2022-07-01 09:02:00  2911.0  0.0   PM2209     0.0

复权价 close/K = [inf, inf, inf]


## B-2 · 主力合约判定在交易日内部乱跳

正常品种的 `contract` 只在**交易日边界**变更。这几个品种的主力合约在盘中反复切换，
等价于价格序列每分钟都在不同合约之间跳。

In [20]:
ctr_chg = np.r_[False, ctr[1:] != ctr[:-1]]
td_chg  = np.r_[False, td[1:]  != td[:-1]]
sym_chg = np.r_[True,  sc[1:]  != sc[:-1]]
intraday = ctr_chg & ~td_chg & ~sym_chg          # 同品种、同交易日内换了合约

b2 = pd.DataFrame({"sym": SYMV, "变更": ctr_chg & ~sym_chg, "盘中变更": intraday}) \
       .groupby("sym").sum().astype(int)
b2["合约数"] = pd.DataFrame({"sym": SYMV, "c": ctr}).groupby("sym").c.nunique()
hitb2 = b2[b2.盘中变更 > 0].sort_values("盘中变更", ascending=False)
print(f"[B-2] 全库 contract 变更 {int(b2.变更.sum()):,} 次, "
      f"其中盘中变更 {int(b2.盘中变更.sum()):,} 次 ({b2.盘中变更.sum()/b2.变更.sum():.2%})")
print(hitb2.to_string())
print(f"\n盘中变更为 0 的品种: {len(b2) - len(hitb2)} / {len(b2)}")
print("池内(59)是否有盘中换月:", sorted(set(hitb2.index) & POOL) or "无")

[B-2] 全库 contract 变更 848,065 次, 其中盘中变更 842,548 次 (99.35%)
         变更    盘中变更  合约数
sym                     
ZC   242746  242430   56
JR   179221  178891   38
WH   178222  177893   53
LR   169485  169213   29
PM    37404   37160   47
RI    37185   36961   45

盘中变更为 0 的品种: 73 / 79
池内(59)是否有盘中换月: 无


In [21]:
# 案例: ZC 某一天, contract 一分钟一换
ex = df[(sym == "ZC") & (df.trading_date == "2015-06-01")]
print("ZC td=2015-06-01 前 12 根:")
print(ex[["close","volume","contract"]].head(12).to_string())
print("\n该交易日出现的合约数:", ex.contract.nunique())

ZC td=2015-06-01 前 12 根:
                     close  volume contract
datetime                                   
2015-06-01 09:00:00  427.4    62.0   TC1509
2015-06-01 09:01:00  427.0   575.0   TC1509
2015-06-01 09:02:00  427.2    82.0   TC1509
2015-06-01 09:03:00  427.0   151.0   TC1509
2015-06-01 09:04:00  427.0    55.0   TC1509
2015-06-01 09:05:00  427.2   204.0   TC1509
2015-06-01 09:06:00  427.2   111.0   TC1509
2015-06-01 09:07:00  426.4   184.0   TC1509
2015-06-01 09:08:00  426.4    59.0   TC1509
2015-06-01 09:09:00  426.2    46.0   TC1509
2015-06-01 09:10:00  426.4    33.0   TC1509
2015-06-01 09:11:00  426.2    37.0   TC1509

该交易日出现的合约数: 1


## B-3 · `(underlying_symbol, datetime)` 时间戳重复

In [22]:
key     = pd.MultiIndex.from_arrays([sc, dt.to_numpy()])
dup_any = key.duplicated()
b3 = pd.Series(SYMV[dup_any]).value_counts()
print(f"[B-3] 重复行合计: {int(dup_any.sum()):,}")
print(b3.to_string() if len(b3) else "无")
print(f"\n重复为 0 的品种: {79 - len(b3)} / 79")
print("池内(59)是否有重复:", sorted(set(b3.index) & POOL) or "无")

[B-3] 重复行合计: 686,240
ZC    206216
JR    147804
WH    141250
LR    132888
PM     29154
RI     28928

重复为 0 的品种: 73 / 79
池内(59)是否有重复: 无


In [23]:
# 案例: 把某个重复时间戳的全部行打出来
ex = df[sym == "JR"]
d2 = ex.index[ex.index.duplicated(keep=False)]
if len(d2):
    t = d2[0]
    print(f"JR 在 {t} 的全部行:")
    print(ex.loc[[t]][["open","high","low","close","volume","contract","K"]].to_string())

JR 在 2024-07-02 09:00:00 的全部行:
                       open    high     low   close  volume contract   K
datetime                                                                
2024-07-02 09:00:00  2662.0  2662.0  2662.0  2662.0     0.0   JR2503 NaN
2024-07-02 09:00:00  2662.0  2662.0  2662.0  2662.0     0.0   JR2411 NaN


## B-4 · 同一 `contract` 内 `K` 取值多于一个

**复权因子按合约恒定是定义要求** —— 同一个合约在其存续期内 K 必须是常数，
变了就说明复权链在合约中途被改写过。

> 文档里写的是「池内零违规」。下面把**每一个**违规合约连同它的两个 K 值、
> 发生日期、变更点前后的原始 bar 一起打出来，不是只报一个 0。

In [24]:
# 逐品种做: 在 6,600 万行 x 两个 object 列上直接 groupby 会吃掉几个 GB
_parts = []
for _i, _sname in enumerate(SYMS):
    _m = sc == _i
    _g = (pd.DataFrame({"contract": ctr[_m], "K": K[_m], "td": td[_m]})
            .groupby("contract").agg(
                K取值数=("K","nunique"), K最小=("K","min"), K最大=("K","max"),
                起=("td","min"), 止=("td","max"), bar数=("K","size")).reset_index())
    _g.insert(0, "sym", _sname)
    _parts.append(_g)
b4 = pd.concat(_parts, ignore_index=True)
del _parts
viol = b4[b4.K取值数 > 1]
print(f"[B-4] 合约总数 {len(b4):,}, 其中 K 非常数的 {len(viol)} 个, 涉及品种 {viol.sym.nunique()} 个")
print(viol.to_string(index=False) if len(viol) else "(无)")
print(f"\n池内(59) 违规合约数: {len(viol[viol.sym.isin(POOL)])}")
print("池内违规品种:", sorted(set(viol.sym) & POOL) or "无 —— 池内零违规")

[B-4] 合约总数 4,041, 其中 K 非常数的 7 个, 涉及品种 2 个
sym contract  K取值数      K最小      K最大          起          止  bar数
 BB   BB2601     2 0.465042 0.470975 2025-04-09 2025-12-01 18984
 BB   BB2603     2 0.465042 0.470975 2025-04-08 2026-01-16 10848
 WR   WR2601     2 0.800461 0.828727 2025-07-03 2025-12-31 13108
 WR   WR2606     2 0.800461 0.828727 2025-08-28 2025-10-31  1130
 WR   WR2607     2 0.798349 0.828727 2025-11-14 2026-03-10  8362
 WR   WR2609     2 0.798349 0.812389 2026-01-13 2026-03-16  1808
 WR   WR2701     2 0.798349 0.825937 2026-03-06 2026-07-29 20566

池内(59) 违规合约数: 0
池内违规品种: 无 —— 池内零违规


In [25]:
# 把违规合约的 K 变更点原样打出来, 确认不是统计口径造成的假象
for _, r in viol.iterrows():
    ex  = df[(sym == r.sym) & (df.contract == r.contract)]
    kk  = ex["K"].to_numpy()
    chg = ex.index[np.r_[False, kk[1:] != kk[:-1]]]
    print(f"--- {r.sym} / {r.contract}: K 在 {len(chg)} 个时点变化, 首个变更点 {chg[0]}")
    i = ex.index.get_loc(chg[0])
    i = i if isinstance(i, int) else int(np.flatnonzero(i)[0])
    print(ex.iloc[max(0,i-2):i+3][["close","volume","contract","K","trading_date"]].to_string())

--- BB / BB2601: K 在 1 个时点变化, 首个变更点 2025-08-29 09:00:00
                      close  volume contract         K trading_date
datetime                                                           
2025-08-25 14:59:00  147.40     0.0   BB2601  0.465042   2025-08-25
2025-08-25 15:00:00  147.40     0.0   BB2601  0.465042   2025-08-25
2025-08-29 09:00:00  146.85     0.0   BB2601  0.470975   2025-08-29
2025-08-29 09:01:00  146.85     0.0   BB2601  0.470975   2025-08-29
2025-08-29 09:02:00  146.85     0.0   BB2601  0.470975   2025-08-29
--- BB / BB2603: K 在 1 个时点变化, 首个变更点 2025-09-01 09:00:00
                      close  volume contract         K trading_date
datetime                                                           
2025-08-18 14:59:00  146.25     0.0   BB2603  0.465042   2025-08-18
2025-08-18 15:00:00  146.25     0.0   BB2603  0.465042   2025-08-18
2025-09-01 09:00:00  147.00     0.0   BB2603  0.470975   2025-09-01
2025-09-01 09:01:00  147.00     0.0   BB2603  0.470975   2025-09-01
2025

## ~~B-5~~ → E-11 · `contract` 的字母前缀 ≠ `underlying_symbol`（**核实后：不是错误**）

郑商所若干品种改过代码（`RO→OI`、`ME→MA`、`TC→ZC`、`WS→WH`、`WT→PM`、`ER→RI`），
拼接后的历史段里 `contract` 留的还是旧代码。

> **向上游核实的结果：这不是缺陷。** txt 层里 **`ME1505` 与 `MA1506` 是同时存在的**
> （`RO1305` 与 `OI1307` 同理）—— 交易所改代码时**新老合约并行挂牌**，老合约挂到自己到期为止。
> vendor 原样透传，pkl 也原样透传，三层一致。
>
> **所以不要修数据，改代码**：禁止用 `contract` 的字母前缀反解品种（一律用 `underlying_symbol`），
> 解析到期月只取末 4 位。下面这格保留，用来确认波及范围。

In [26]:
pairs = pd.DataFrame({"sym": SYMV, "contract": ctr}).drop_duplicates()
pairs["前缀"] = pairs.contract.str.extract(r"^([A-Za-z]+)")[0].str.upper()
badc  = pairs[pairs.前缀 != pairs.sym]
cnt   = pd.DataFrame({"sym": SYMV, "contract": ctr, "td": td})
cnt   = cnt[cnt.contract.isin(set(badc.contract))].groupby("sym").agg(
            bar数=("td","size"), 起=("td","min"), 止=("td","max"))
print(f"[B-5] 前缀不符的合约: {len(badc)} 个, 涉及品种 {badc.sym.nunique()} 个")
out = badc.groupby("sym").agg(合约数=("contract","size"),
        旧代码=("前缀", lambda x: sorted(set(x))),
        合约=("contract", lambda x: sorted(x))).join(cnt)
print(out[["合约数","旧代码","bar数","起","止"]].to_string())
print("\n其中在池内(59)的:", sorted(set(badc.sym) & POOL))
print("\n完整合约码:")
for s, r in out.iterrows():
    print(f"  {s}: {r.合约}")

[B-5] 前缀不符的合约: 55 个, 涉及品种 6 个
     合约数   旧代码    bar数          起          止
sym                                         
MA    10  [ME]  170178 2011-11-03 2014-12-10
OI     9  [RO]  176958 2010-01-04 2013-03-27
PM    10  [WT]  146448 2010-01-04 2012-08-31
RI    10  [ER]  179670 2010-01-04 2013-04-16
WH     9  [WS]  178766 2010-01-04 2013-04-10
ZC     7  [TC]  115712 2013-10-09 2015-11-10

其中在池内(59)的: ['MA', 'OI']

完整合约码:
  MA: ['ME1203', 'ME1205', 'ME1209', 'ME1301', 'ME1305', 'ME1309', 'ME1401', 'ME1405', 'ME1409', 'ME1501']
  OI: ['RO1009', 'RO1101', 'RO1105', 'RO1109', 'RO1201', 'RO1205', 'RO1209', 'RO1301', 'RO1305']
  PM: ['WT1005', 'WT1009', 'WT1011', 'WT1101', 'WT1103', 'WT1105', 'WT1109', 'WT1201', 'WT1205', 'WT1209']
  RI: ['ER1005', 'ER1009', 'ER1101', 'ER1105', 'ER1109', 'ER1201', 'ER1205', 'ER1209', 'ER1301', 'ER1305']
  WH: ['WS1009', 'WS1101', 'WS1105', 'WS1109', 'WS1201', 'WS1205', 'WS1209', 'WS1301', 'WS1305']
  ZC: ['TC1401', 'TC1405', 'TC1409', 'TC1501', 'TC1505', 'TC1

In [27]:
# 案例: MA 在 ME -> MA 的换代码点
ex = df[sym == "MA"]
ex = ex[(ex.trading_date >= "2014-12-08") & (ex.trading_date <= "2014-12-12")]
print("MA 换代码前后, 每个交易日出现的 contract:")
print(ex.groupby(ex.trading_date.dt.strftime("%Y-%m-%d")).contract.unique().to_string())

MA 换代码前后, 每个交易日出现的 contract:
trading_date
2014-12-08    [ME1501]
2014-12-09    [ME1501]
2014-12-10    [ME1501]
2014-12-11    [MA1506]
2014-12-12    [MA1506]


---
# C 类 · 交付缺口

pkl 少了 txt 层有的东西。**这两条要去 txt 层核**（`data/00-跨层/compare_layers.py`），
pkl 内部只能确认「缺了」，不能确认「上游有没有」。

## C-1 · 9 个品种整体止于 2024-04-22

判据不是「停更」而是「**停更当天还有没有人持仓**」——
没有人会把几万手持仓留到品种消失，所以末日持仓量是区分「品种死了」和「文件断了」的唯一硬证据。

In [28]:
# 用整数品种码分组 (object 列分组在 6,600 万行上很贵), 最后再映射回名字
DAILY = (pd.DataFrame({"sc": sc, "td": td, "vol": vol, "oi": oi})
           .groupby(["sc","td"]).agg(vol=("vol","sum"), oi_last=("oi","last")).reset_index())
DAILY["sym"] = SYMS[DAILY.pop("sc").to_numpy()]
last_td = DAILY.groupby("sym").td.max()
stale   = last_td[last_td < DAILY.td.max() - np.timedelta64(30, "D")]

rows = []
for s in stale.index:
    g = DAILY[DAILY.sym == s].sort_values("td")
    rows.append(dict(sym=s, 首日=str(g.td.min())[:10], 末日=str(g.td.max())[:10],
                     末日OI=int(g.oi_last.iloc[-1]), 常态OI中位=int(g.oi_last.median()),
                     比值=round(g.oi_last.iloc[-1] / max(g.oi_last.median(), 1), 3),
                     末5日量比=round(g.vol.tail(5).mean() / max(g.vol.median(), 1), 3)))
print("[C-1] 全部停更品种:")
print(pd.DataFrame(rows).sort_values(["末日","sym"]).to_string(index=False))
print("\n判读: 末日 OI 仍在万手量级 -> 数据截断(市场活着); 末日 OI = 0 -> 品种真的死了")
print("注意 BB(92) / FB(1479) / RS(2) 的 OI 并没有归零 ——")
print("     它们的剔除理由只能是流动性, 不是 v3.0-LEDG 写的「停更且末日持仓归零」")

[C-1] 全部停更品种:
sym         首日         末日   末日OI  常态OI中位     比值  末5日量比
 AO 2023-06-27 2024-04-22  59673   63196  0.944  1.589
 BR 2023-08-03 2024-04-22  11628   34364  0.338  0.723
 EC 2023-08-24 2024-04-22  27622   31606  0.874  0.364
 IM 2022-07-28 2024-04-22 110846   69433  1.596  3.953
 LC 2023-07-27 2024-04-22 177912  136355  1.305  0.727
 PX 2023-09-21 2024-04-22  76971   79312  0.970  1.348
 SH 2023-09-21 2024-04-22  26932   51362  0.524  0.627
 SI 2022-12-28 2024-04-22 147074   69179  2.126  3.670
 TL 2023-04-27 2024-04-22  61898   31081  1.992  1.841
 LR 2014-07-14 2025-12-31      0       0  0.000  0.000
 BB 2013-12-12 2026-01-16     92       1 92.000  6.800
 FB 2013-12-12 2026-01-16   1479     667  2.217  0.599
 JR 2013-11-22 2026-01-16      0       1  0.000  0.000
 PM 2010-01-04 2026-01-16      0       4  0.000  0.000
 RI 2010-01-04 2026-01-16      0      13  0.000  0.000
 RS 2013-01-08 2026-01-16      2      12  0.167  2.840
 WH 2010-01-04 2026-01-16      0    2531  0.000  0.

## C-2 · 品种在自身存活区间内缺交易日

In [29]:
mkt_days = set(pd.Timestamp(x) for x in MKT_DAY[MKT_DAY > 0].index)
rows = []
for s, g in DAILY.groupby("sym"):
    have = set(pd.Timestamp(x) for x in g.td)
    a, b = min(have), max(have)
    miss = sorted(d for d in mkt_days if a <= d <= b and d not in have)
    rows.append(dict(sym=s, 首日=f"{a:%Y-%m-%d}", 末日=f"{b:%Y-%m-%d}",
                     有=len(have), 缺=len(miss),
                     缺的日子=", ".join(f"{d:%Y-%m-%d}" for d in miss[:5]) + ("..." if len(miss) > 5 else "")))
C2 = pd.DataFrame(rows)
print(f"[C-2] 市场交易日历共 {len(mkt_days):,} 天")
print(f"      零缺失的品种: {int((C2.缺 == 0).sum())} / {len(C2)}")
print(C2[C2.缺 > 0].to_string(index=False))

[C-2] 市场交易日历共 4,017 天
      零缺失的品种: 77 / 79
sym         首日         末日    有  缺                                                          缺的日子
 FB 2013-12-12 2026-01-16 2907 34 2019-10-15, 2019-10-16, 2019-10-17, 2019-10-18, 2019-10-21...
 LR 2014-07-14 2025-12-31 2775 14 2020-07-01, 2020-07-02, 2020-07-03, 2020-07-06, 2020-07-07...


### C-2 的方法盲区 —— 必须用外部日历补一刀

上面那格的「市场交易日历」是**从 pkl 自己推出来的**（全库任一品种当日日盘有成交的日历日）。
这个口径查得出「某品种相对其他品种缺哪天」，但**查不出「pkl 整体缺哪天」** ——
整库都没有的那天，根本不会进这个日历，于是永远看不见。

补法：拿 txt 层的日频文件名当外部日历（`future_pricemin{YYYYMMDD}.txt`，一个交易日一个文件）。

In [30]:
import glob, re
TXT = os.path.join(HERE, "..", "..", "02-txt层", "原始数据")
files = glob.glob(os.path.join(TXT, "future_pricemin*.txt"))
if not files:
    print("找不到 txt 层, 跳过。手动把 TXT 指到 data/02-txt层/原始数据")
else:
    txt_days = {pd.Timestamp(m.group(1)) for f in files
                for m in [re.search(r"future_pricemin(\d{8})\.txt$", os.path.basename(f))] if m}
    pkl_days = {pd.Timestamp(x) for x in pd.unique(td)}
    # 注意别用 lo / hi 当变量名 —— 那是全局的 low / high 价格数组
    d_lo, d_hi = max(min(txt_days), min(pkl_days)), min(max(txt_days), max(pkl_days))
    only_txt = sorted(d for d in txt_days - pkl_days if d_lo <= d <= d_hi)
    only_pkl = sorted(d for d in pkl_days - txt_days if d_lo <= d <= d_hi)
    print(f"txt 日频文件 {len(txt_days):,} 个, pkl trading_date {len(pkl_days):,} 个, "
          f"重叠区间 {d_lo:%Y-%m-%d} ~ {d_hi:%Y-%m-%d}")
    print("")
    print(f"[C-2 补] txt 有文件、pkl 整库没有的交易日: {len(only_txt)} 天")
    for d in only_txt:
        sz = os.path.getsize(os.path.join(TXT, f"future_pricemin{d:%Y%m%d}.txt")) / 1e6
        print(f"   {d:%Y-%m-%d}  txt 文件 {sz:.1f} MB  <- pkl 整库缺这一天, 从 pkl 内部看不出来")
    print("")
    head = ", ".join(f"{d:%Y-%m-%d}" for d in only_pkl[:10]) + (" ..." if len(only_pkl) > 10 else "")
    print(f"     pkl 有、txt 没有的交易日: {len(only_pkl)} 天   {head}")
    print("")
    print("注意: txt 文件名只是外部日历的一个近似 —— 它也可能自己缺文件(已知 43 天),")
    print("      所以 only_pkl 那一列不等于 pkl 多造了数据, 要去 v3.2 的跨层对账看。")

txt 日频文件 3,989 个, pkl trading_date 4,017 个, 重叠区间 2010-01-04 ~ 2026-07-29

[C-2 补] txt 有文件、pkl 整库没有的交易日: 3 天
   2025-09-10  txt 文件 23.9 MB  <- pkl 整库缺这一天, 从 pkl 内部看不出来
   2026-02-12  txt 文件 24.5 MB  <- pkl 整库缺这一天, 从 pkl 内部看不出来
   2026-02-13  txt 文件 24.6 MB  <- pkl 整库缺这一天, 从 pkl 内部看不出来

     pkl 有、txt 没有的交易日: 39 天   2022-09-26, 2023-06-12, 2023-06-13, 2023-06-14, 2023-06-15, 2023-06-16, 2023-06-19, 2023-06-20, 2023-06-21, 2023-06-26 ...

注意: txt 文件名只是外部日历的一个近似 —— 它也可能自己缺文件(已知 43 天),
      所以 only_pkl 那一列不等于 pkl 多造了数据, 要去 v3.2 的跨层对账看。


---
# D 类 · 数值硬错误

量极小（合计不到万分之一），**直接丢弃这些 bar，不要修补**。

## D-1 · `total_turnover` / `volume` / `open_interest` 出现负值

In [31]:
neg_any = (tov < 0) | (vol < 0) | (oi < 0)
d1 = pd.DataFrame({"sym": SYMV,
                   "负成交额": tov < 0, "负成交量": vol < 0, "负持仓": oi < 0}) \
       .groupby("sym").sum().astype(int)
d1 = d1[d1.sum(axis=1) > 0].sort_values("负成交额", ascending=False)
inpool = np.isin(SYMV, list(POOL))
print(f"[D-1] 负值 bar 合计 {int(neg_any.sum()):,} 根, 涉及 {len(d1)} 个品种")
print(d1.assign(在池内=[s in POOL for s in d1.index]).to_string())
print(f"\n池内(59) 负值 bar: {int(neg_any[inpool].sum()):,} 根")
print("上游对应: txt 层 P1-7「郑商所成交额凑整万元 -> 负成交额」145,887 根")
print()
print(df[tov < 0][["close","volume","total_turnover","open_interest",
                   "underlying_symbol","contract"]].head(5).to_string())

[D-1] 负值 bar 合计 4,160 根, 涉及 21 个品种
     负成交额  负成交量  负持仓    在池内
sym                        
RS   1236     0    0  False
WH    608     0    0  False
RI    585     0    0  False
JR    469     0    0  False
LR    287     0    0  False
CY    273     0    0   True
PM    212     0    0  False
SF    192     0    0   True
SM    112     0    0   True
ZC    105     0    0  False
PF     15     0    0   True
SA     15     0    0   True
PX     12     0    0  False
SR      9     0    0   True
PK      7     0    0   True
OI      5     0    0   True
CF      5     0    0   True
TA      5     0    0   True
MA      4     0    0   True
RM      3     0    0   True
PB      1     1    0   True

池内(59) 负值 bar: 646 根
上游对应: txt 层 P1-7「郑商所成交额凑整万元 -> 负成交额」145,887 根

                       close  volume  total_turnover  open_interest underlying_symbol contract
datetime                                                                                      
2023-12-26 13:42:00  15450.0   104.0      -4296550.0       653

## D-2 · 量额矛盾（一个为 0 另一个非 0）

In [32]:
mix_a = (vol == 0) & (tov > 0)      # 没成交却有成交额
mix_b = (vol >  0) & (tov == 0)     # 有成交却没成交额
d2 = pd.DataFrame({"sym": SYMV, "量0额非0": mix_a, "量非0额0": mix_b}) \
       .groupby("sym").sum().astype(int)
d2 = d2[d2.sum(axis=1) > 0]
print(f"[D-2] 合计 {int((mix_a | mix_b).sum()):,} 根")
print(d2.assign(在池内=[s in POOL for s in d2.index]).to_string())
print("\n注意 RU 的方向和别人相反 (有成交量却没有成交额):")
print(df[mix_b][["close","volume","total_turnover","underlying_symbol","contract"]].head(3).to_string())

[D-2] 合计 266 根
     量0额非0  量非0额0    在池内
sym                     
CY     100      0   True
JR       1      0  False
LR       6      0  False
PM       1      0  False
RI      11      0  False
RS      57      0  False
RU       0     50   True
SF       3      0   True
WH      11      0  False
ZC      26      0  False

注意 RU 的方向和别人相反 (有成交量却没有成交额):
                       close  volume  total_turnover underlying_symbol contract
datetime                                                                       
2010-12-20 10:42:00  37440.0   102.0             0.0                RU   RU1105
2010-12-20 10:43:00  37440.0   272.0             0.0                RU   RU1105
2010-12-20 10:44:00  36830.0   135.0             0.0                RU   RU1105


## D-3 · OHLC ≤ 0 或 NaN

In [33]:
bad_px = (np.isnan(op) | np.isnan(hi) | np.isnan(lo) | np.isnan(cl)
          | (op <= 0) | (hi <= 0) | (lo <= 0) | (cl <= 0))
d3 = pd.DataFrame({"sym": SYMV, "坏价": bad_px}).groupby("sym").坏价.sum()
d3 = d3[d3 > 0].astype(int)
print(f"[D-3] OHLC<=0 或 NaN 合计 {int(bad_px.sum())} 根")
print(d3.to_string())
print(f"其中 OHLC 含 NaN 的: "
      f"{int((np.isnan(op)|np.isnan(hi)|np.isnan(lo)|np.isnan(cl)).sum())} 根")
print("")
print("上游: txt 层同形 (P1-9「时段首笔成交 OHLC 被写成 0」438,756 行),")
print("到 pkl 只剩 96 根 —— 88% 的坏块落在非主力合约上, 被主力筛选滤掉了。Level-1 未查, 不作断言。")
print("处置: 坏价只出现在 B / BB / RR 三个品种上, 都是小品种 -> 整体剔除, 不做逐 bar 修补。")
print()
print(df[bad_px][["open","high","low","close","volume",
                  "underlying_symbol","trading_date"]].head(5).to_string())

[D-3] OHLC<=0 或 NaN 合计 96 根
sym
B     91
BB     1
RR     4
其中 OHLC 含 NaN 的: 0 根

上游: txt 层同形 (P1-9「时段首笔成交 OHLC 被写成 0」438,756 行),
到 pkl 只剩 96 根 —— 88% 的坏块落在非主力合约上, 被主力筛选滤掉了。Level-1 未查, 不作断言。
处置: 坏价只出现在 B / BB / RR 三个品种上, 都是小品种 -> 整体剔除, 不做逐 bar 修补。

                     open  high  low  close  volume underlying_symbol trading_date
datetime                                                                          
2017-05-12 13:31:00   0.0   0.0  0.0    0.0     1.0                 B   2017-05-12
2017-05-12 13:32:00   0.0   0.0  0.0    0.0     1.0                 B   2017-05-12
2017-05-12 13:33:00   0.0   0.0  0.0    0.0     0.0                 B   2017-05-12
2017-05-12 13:34:00   0.0   0.0  0.0    0.0     0.0                 B   2017-05-12
2017-05-12 13:35:00   0.0   0.0  0.0    0.0     0.0                 B   2017-05-12


---
# F 类 · 集合竞价

这两条**只有 Level-1 能定性**：pkl 侧只看得到「少了一根 bar」和「OHLC 不相等」两个表象，
定不了「成交去哪了」和「哪个价才是真的」。下面几格只做一件事 —— 把波及范围量清楚。

## F-1 · 日盘集合竞价没有独立 bar，成交被并进相邻 bar

2023-05-26 起上期所 / 大商所 / 上期能源开放日盘集合竞价。Level-1 是**累计**口径，设某 trading_day 上

- A = 夜盘倒数第二分钟 tick（02:30 收盘品种即 02:29:00）
- B = 夜盘最后一 tick（02:30:00）
- C = 日盘集合竞价 08:59:00
- D = 09:00:00
- E = 09:01:00

现在的归属是错的：

- **23:00 收盘品种**：夜盘最后一根 = B−A，09:01 那根 = E−B → 日盘竞价**完全并入**日盘第一根
- **01:00 / 02:30 收盘品种**：夜盘最后一根 = C−A，09:01 那根 = E−C → 日盘竞价被**拆分**给两根

应该分成三组：**B−A**（夜盘最后一根）、**D−B**（日盘集合竞价，单独一根 end 09:00）、**E−D**（日盘第一根）。

郑商所虽在同一新规内，但只开 08:55–08:59 撤单窗口、不能挂单，**没有日盘集合竞价数据**，
不要为郑商所生成这根 bar。

pkl 侧能验的只有一件事：**这根 bar 到底存不存在。**

In [34]:
CUT  = np.datetime64("2023-05-26")
post = td >= CUT
_rows = []
for _i, _s in enumerate(SYMS):
    m = (sc == _i) & post
    if not m.any():
        continue
    _rows.append(dict(
        sym=str(_s), ex=EXCH[str(_s)],
        有夜盘=bool(((mod[m] >= 1260) | (mod[m] <= 179)).any()),
        交易日=int(np.unique(td[m]).size),
        有09_00的日子=int(np.unique(td[m & (mod == 540)]).size)))
F1 = pd.DataFrame(_rows)

print("=== 2023-05-26 起, 有夜盘品种的 09:00 日盘竞价 bar ===")
print(F1[F1.有夜盘].groupby("ex").agg(品种数=("sym","size"), 交易日合计=("交易日","sum"),
      有09_00的日子=("有09_00的日子","sum")).to_string())
_t = F1[F1.有夜盘 & F1.ex.isin(["SHFE","DCE","INE"])]
print(f"新规覆盖的三家合计: {len(_t)} 个品种 / {int(_t.交易日.sum()):,} 个(品种x交易日), "
      f"其中有 09:00 bar 的天数: {int(_t.有09_00的日子.sum())}")
print("")
print("=== 对照: 无夜盘品种本来就有 09:00 竞价 bar ===")
print(F1[~F1.有夜盘].groupby("ex").agg(品种数=("sym","size"), 交易日合计=("交易日","sum"),
      有09_00的日子=("有09_00的日子","sum")).to_string())
print("")
print("注: 中金所竞价在 09:30 不在 09:00, 所以它在对照组里是 0, 属正常。")

=== 2023-05-26 起, 有夜盘品种的 09:00 日盘竞价 bar ===
      品种数  交易日合计  有09_00的日子
ex                         
DCE    17  12988          0
INE     4   3056          0
SHFE   17  11833          0
ZCE    13   8557          0
新规覆盖的三家合计: 38 个品种 / 27,877 个(品种x交易日), 其中有 09:00 bar 的天数: 0

=== 对照: 无夜盘品种本来就有 09:00 竞价 bar ===
       品种数  交易日合计  有09_00的日子
ex                          
CFFEX    8   5024          0
DCE      4   2810       2810
GFEX     2    398        398
INE      1    158        158
SHFE     1    764        764
ZCE     12   8420       8420

注: 中金所竞价在 09:30 不在 09:00, 所以它在对照组里是 0, 属正常。


## F-2 · 郑商所集合竞价 bar 不是平 K 线

集合竞价是单一撮合价，这根 bar 的 OHLC 应当四价相等。郑商所的不相等。
回查 Level-1 后确认：**这根 bar 的 `open` 对得上真实撮合价**，脏的是 high / low / close。
要求数据端用现有的 `open` 生成平 K 线（`high = low = close = open`），量与持仓不动。

> **口径陷阱（第一版就栽在这，留作反面教材）**：**不能用固定时刻取竞价 bar。**
> 中金所 2016-01-04 之前开盘是 **09:15** 不是 09:30，大商所夜盘是 2013–2014 分批引入的 ——
> 按固定偏移取，会把一堆普通盘中 bar 当成竞价 bar，
> 能把中金所的非平 K 根数从真实的 **390** 放大到 **4,618**。
>
> 正确定义是**「交易日第一根 bar」**：有夜盘就是 21:00 那根，没夜盘就是 09:00 / 09:15 / 09:30 那根，
> 不用关心它具体几点。再要求 `volume > 0` —— 零成交的 bar 平不平没有意义。

In [35]:
# 竞价 bar = 交易日第一根 bar。数据已按 品种 -> 时间 分块排序, 所以相邻比较即可
_first = np.r_[True, (td[1:] != td[:-1]) | (sc[1:] != sc[:-1])]
_sel   = _first & (vol > 0)
_flat  = (op == hi) & (hi == lo) & (lo == cl)

F2 = pd.DataFrame({
    "sym":  SYMV[_sel],
    "ex":   [EXCH[str(x)] for x in SYMV[_sel]],
    "td":   td[_sel],
    "yr":   pd.DatetimeIndex(td[_sel]).year,
    "flat": _flat[_sel],
    "open": op[_sel], "high": hi[_sel], "low": lo[_sel], "close": cl[_sel],
})
print(f"竞价 bar 样本 (交易日第一根且有成交): {len(F2):,}")
print("")
print("=== 平 K 比例, 按交易所 ===")
print(F2.groupby("ex").agg(bar数=("flat","size"), 平K比例=("flat","mean"),
      非平K根数=("flat", lambda x: int((~x).sum()))).assign(
      平K比例=lambda d: d.平K比例.round(4)).to_string())
print("")
print("=== 非平 K 的年份分布 ===")
print(pd.crosstab(F2[~F2.flat].ex, F2[~F2.flat].yr).to_string())

竞价 bar 样本 (交易日第一根且有成交): 173,978

=== 平 K 比例, 按交易所 ===
        bar数    平K比例  非平K根数
ex                         
CFFEX  16023  0.9757    390
DCE    57203  0.9759   1380
GFEX     492  1.0000      0
INE     6430  1.0000      0
SHFE   46878  0.9997     15
ZCE    46952  0.2516  35137

=== 非平 K 的年份分布 ===
yr     2010  2011  2012  2013  2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025  2026
ex                                                                                                         
CFFEX     0     0    51   206   133     0     0     0     0     0     0     0     0     0     0     0     0
DCE       0     0     0     8  1371     0     0     0     0     0     0     0     1     0     0     0     0
SHFE      5     7     0     0     0     0     0     0     0     0     0     0     0     1     2     0     0
ZCE     453   152   340   356  1261  1412  1742  2034  2483  2714  3103  3528  3343  3400  3492  3421  1903


In [36]:
Z = F2[F2.ex == "ZCE"]
print("=== 郑商所逐年平 K 比例 —— 在持续恶化 ===")
print(Z.groupby("yr").agg(bar数=("flat","size"), 平K比例=("flat","mean")).assign(
      平K比例=lambda d: d.平K比例.round(3)).T.to_string())
print("")

NF = Z[~Z.flat].copy()
NF["range_bp"]      = (NF["high"] - NF["low"]) / NF["open"] * 1e4
NF["close_open_bp"] = (NF["close"] - NF["open"]).abs() / NF["open"] * 1e4
print(f"=== 失真幅度 (基点), n = {len(NF):,} ===")
for _c in ["range_bp", "close_open_bp"]:
    _q = NF[_c].quantile([.5, .9, .99])
    print(f"  {_c:15s} 中位 {_q[.5]:7.1f}   p90 {_q[.9]:8.1f}   p99 {_q[.99]:9.1f}")
print("  -> 把这根 bar 的 close 当开盘价用, 系统性偏 6-7 个基点, 比手续费还大")
print("")

ZP = sorted(set(map(str, Z.sym)) & POOL)
print(f"=== 池内郑商所 {len(ZP)} 个品种, 2024 年起 ===")
_r = Z[(Z.yr >= 2024) & (Z.sym.isin(ZP))].groupby("sym").agg(
        bar数=("flat","size"), 平K比例=("flat","mean"))
_r["非平K"] = _r.bar数 - (_r.bar数 * _r.平K比例).round().astype(int)
print(_r.assign(平K比例=lambda d: d.平K比例.round(3)).sort_values("平K比例").to_string())
print(f"合计非平K: {int(_r.非平K.sum()):,} / {int(_r.bar数.sum()):,}")
print("")

print("=== 大商所 / 中金所: 只登记不交办 ===")
print("    (2015 年起两家都已干净, 纯历史遗留; 且 Level-1 不覆盖 2012-2014, 无法仲裁)")
for _e in ["DCE", "CFFEX"]:
    _x = F2[(F2.ex == _e) & (~F2.flat)]
    print(f"  {_e}: {len(_x)} 根, {_x.td.min()} ~ {_x.td.max()}, {_x.td.nunique()} 个交易日")
    print("     品种: " + " ".join(f"{k}({v})" for k, v in _x.sym.value_counts().items()))

=== 郑商所逐年平 K 比例 —— 在持续恶化 ===
yr        2010      2011      2012      2013      2014      2015      2016      2017      2018      2019      2020      2021      2022      2023      2024      2025      2026
bar数  1404.000  1477.000  1531.000  1841.000  2328.000  2056.000  2245.000  2470.000  2964.000  3107.000  3699.000  4126.000  3940.000  3945.000  3935.000  3805.000  2079.000
平K比例     0.677     0.897     0.778     0.807     0.458     0.313     0.224     0.177     0.162     0.126     0.161     0.145     0.152     0.138     0.113     0.101     0.085

=== 失真幅度 (基点), n = 35,137 ===
  range_bp        中位     7.6   p90     20.6   p99      46.1
  close_open_bp   中位     6.7   p90     19.5   p99      44.4
  -> 把这根 bar 的 close 当开盘价用, 系统性偏 6-7 个基点, 比手续费还大

=== 池内郑商所 16 个品种, 2024 年起 ===
     bar数   平K比例  非平K
sym                  
SM    616  0.042  590
OI    598  0.045  571
AP    616  0.055  582
PF    598  0.060  562
SF    616  0.063  577
SR    598  0.077  552
CJ    616  0.081  566
TA    598  0.082 

---
# Z · 「池内零违规」总自查

文档里凡是声称**为 0** 的地方，这一节全部重跑一遍，并且**把违规行原样打出来**
（真为 0 时打印 `(空)`）。不接受「我算出来是 0」，要看到 0 是怎么来的。

In [37]:
checks = {}
checks["E2 high < low"]                 = int((hi < lo).sum())
checks["E2 high < max(open,close)"]     = int((hi < np.maximum(op, cl)).sum())
checks["E2 low > min(open,close)"]      = int((lo > np.minimum(op, cl)).sum())
checks["时间戳非整分钟(秒!=0)"]         = int((dt.second.to_numpy() != 0).sum())
checks["时间戳非整分钟(微秒!=0)"]       = int((dt.microsecond.to_numpy() != 0).sum())
checks["trading_date 为 NaT"]           = int(df["trading_date"].isna().sum())
checks["trading_date 落在周末"]         = int(pd.DatetimeIndex(td).dayofweek.isin([5,6]).sum())
checks["日历日落在周日"]                = int(pd.DatetimeIndex(cal).dayofweek.isin([6]).sum())
# 注意: df.duplicated() 只看列、不看 index, 会把「不同分钟但 OHLCV 一模一样」的
# padding bar 全算成重复 (7,819,379 根), 那是填充不是重复行。真正的「整行重复」
# 必须把 datetime 一起算进去 —— 即在 (品种,时间戳) 已经重复的那批里, 再看是否逐列相同。
dup_key  = pd.MultiIndex.from_arrays([sc, dt.to_numpy()]).duplicated(keep=False)
dup_full = int(df[dup_key].reset_index().duplicated().sum()) if dup_key.any() else 0
checks["整行完全重复(含 datetime)"]     = dup_full
checks["池内 时间戳重复"]               = int(pd.MultiIndex.from_arrays(
                                             [sc[inpool], dt.to_numpy()[inpool]]).duplicated().sum())
checks["池内 盘中换月"]                 = int(intraday[inpool].sum())
checks["池内 K 为 NaN 或 <=0"]          = int(((np.isnan(K)) | (K <= 0))[inpool].sum())
checks["池内 OHLC 含 NaN"]              = int((np.isnan(op)|np.isnan(hi)|np.isnan(lo)|np.isnan(cl))[inpool].sum())
checks["池内 K 在合约内非常数(合约数)"] = int(len(viol[viol.sym.isin(POOL)]))
# 同一 (品种,交易日) 内出现两个以上合约 == 相邻两行同键但合约码不同 (数据已按 品种,时间 排好序)
_same_key = (sc[1:] == sc[:-1]) & (td[1:] == td[:-1])
_multi    = _same_key & (ctr[1:] != ctr[:-1])
checks["池内 同交易日多个 contract"]    = int((_multi & inpool[1:]).sum())

print("=== 声称为 0 的项, 实测 ===")
for k, v in checks.items():
    print(f"  {'OK  ' if v == 0 else 'HIT '} {k:34s} = {v:,}")

print("")
print("对照 (不是错误, 别误读): 去重时**不带 datetime**, 也就是直接 df.duplicated(),")
print(f"会命中七百多万根 —— 那是 padding bar 在不同分钟重复同一组 OHLCV (平 K 线 {int(pad_bar.sum()):,} 根),")
print("属于 B1 填充现象, 不是「重复行」。判重复必须把 datetime 一起算。")
print("(这里不实跑 df.duplicated(): 它要在 6,600 万行上再开一个 int64 索引, 约 500 MB, 容易 MemoryError)")

=== 声称为 0 的项, 实测 ===
  OK   E2 high < low                      = 0
  OK   E2 high < max(open,close)          = 0
  OK   E2 low > min(open,close)           = 0
  OK   时间戳非整分钟(秒!=0)                      = 0
  OK   时间戳非整分钟(微秒!=0)                     = 0
  OK   trading_date 为 NaT                 = 0
  OK   trading_date 落在周末                  = 0
  OK   日历日落在周日                            = 0
  OK   整行完全重复(含 datetime)                 = 0
  OK   池内 时间戳重复                           = 0
  OK   池内 盘中换月                            = 0
  OK   池内 K 为 NaN 或 <=0                   = 0
  OK   池内 OHLC 含 NaN                      = 0
  OK   池内 K 在合约内非常数(合约数)                  = 0
  OK   池内 同交易日多个 contract                 = 0

对照 (不是错误, 别误读): 去重时**不带 datetime**, 也就是直接 df.duplicated(),
会命中七百多万根 —— 那是 padding bar 在不同分钟重复同一组 OHLCV (平 K 线 8,576,206 根),
属于 B1 填充现象, 不是「重复行」。判重复必须把 datetime 一起算。
(这里不实跑 df.duplicated(): 它要在 6,600 万行上再开一个 int64 索引, 约 500 MB, 容易 MemoryError)


In [38]:
def show(title, mask, cols):
    print(f"### {title}")
    m = np.asarray(mask)
    print(df[m][cols].head(10).to_string() if m.any() else "(空)")
    print()

show("E2 high < low",   hi < lo, ["open","high","low","close","underlying_symbol"])
show("池内 盘中换月",    intraday & inpool, ["close","volume","contract","underlying_symbol","trading_date"])
show("池内 K 非法",      ((np.isnan(K)) | (K <= 0)) & inpool, ["close","K","contract","underlying_symbol"])
print("### 整行完全重复(含 datetime)")
print(df[dup_key].reset_index()[df[dup_key].reset_index().duplicated(keep=False)].head(10).to_string()
      if dup_full else "(空)")

### E2 high < low
(空)

### 池内 盘中换月
(空)

### 池内 K 非法
(空)

### 整行完全重复(含 datetime)
(空)


In [39]:
# session 内缺分钟: 裸 pkl 应为 0 (vendor 只补不漏)
step  = np.diff(mod)
same  = (np.diff(td.astype("int64")) == 0) & (np.diff(sc) == 0)
holes = same & (step > 1) & (step <= 15)   # 上限 15 避开 10:15->10:31 的 16 分钟小节休
print(f"[自查] session 内缺分钟 (间隔 2~15 分钟): {int(holes.sum())} 处, "
      f"缺 {int((step[holes]-1).sum())} 分钟")
if holes.any():
    i = int(np.flatnonzero(holes)[0])
    print(df.iloc[i-1:i+2][["close","volume","underlying_symbol","trading_date"]].to_string())
print("\n含义: 裸 pkl 里 session 内是连续的 —— 之后任何清洗流程打出来的洞, 都是清洗自己造的。")

[自查] session 内缺分钟 (间隔 2~15 分钟): 0 处, 缺 0 分钟

含义: 裸 pkl 里 session 内是连续的 —— 之后任何清洗流程打出来的洞, 都是清洗自己造的。


In [40]:
# 汇总: 四类问题各自要删多少
a1_all  = int(NIGHT[NIGHT.vol == 0].bars.sum())
a1_fab  = int(NIGHT[(NIGHT.vol == 0) & (NIGHT.mkt_vol == 0)].bars.sum())
a2_bars = int(cand.bars.sum())
a3_bars = int(a3.bars.sum())
hard    = int((bad_px | neg_any | mix_a | mix_b).sum())
print("=== 清理量汇总 (全库口径) ===")
print(f"A-1 夜盘整段零成交         {a1_all:>10,} 根  (其中确定伪造 {a1_fab:,}, 其余归 A-4 勿删)")
print(f"A-2 除末根外全零           {a2_bars:>10,} 根  (A 型删整段 / B 型是真数据被毁)")
print(f"A-3 元旦伪造日盘           {a3_bars:>10,} 根")
print(f"D   硬错误(负值/坏价/量额)  {hard:>10,} 根  = 全库 {hard/len(df):.6%}")

=== 清理量汇总 (全库口径) ===
A-1 夜盘整段零成交            431,764 根  (其中确定伪造 109,653, 其余归 A-4 勿删)
A-2 除末根外全零               54,963 根  (A 型删整段 / B 型是真数据被毁)
A-3 元旦伪造日盘               33,300 根
D   硬错误(负值/坏价/量额)       4,522 根  = 全库 0.006822%
